#MINI PROJECT
smart notes search engine

PROJECT OVERVIEW:
Goal:build a serch engine that finds relevent college notes by MEANING,not just keywords
STEPS:
1.create a new chromadb collection for college notes
2.index all 15 notes in the collection
3.run semantic queries and display top results
4.filter results by subject
5.run a comparison keyword search vs semantic search

In [1]:
!pip install chromadb sentence-transformers pandas -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [2]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

In [3]:
df = pd.read_csv("college_notes.csv")

df.head()

,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [4]:
print(df.shape)
print(df.columns)

(15, 4)
Index(['note_id', 'subject', 'topic', 'content'], dtype='object')


In [5]:
#create chromodb collection
client = chromadb.Client()

collection = client.create_collection(
    name="college_notes"
)

print("Collection Created")

Collection Created


In [6]:
#index all notes
collection.add(
    ids=df["note_id"].astype(str).tolist(),
    documents=df["content"].tolist(),
    metadatas=[
        {
            "subject": row["subject"],
            "topic": row["topic"]
        }
        for _, row in df.iterrows()
    ]
)

print("All notes indexed successfully")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 33.2MiB/s]


All notes indexed successfully


In [7]:
print("Total Notes:", collection.count())

Total Notes: 15


In [8]:
#sementic search
query = "How do we collect data from websites and applications?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i, doc in enumerate(results["documents"][0],1):
    print(f"{i}. {doc}")
    print()

1. An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.

2. A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.

3. Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.



In [9]:
#Display Topic and Subject
query = "How do we collect data from websites and applications?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i in range(len(results["documents"][0])):

    print("Subject:",
          results["metadatas"][0][i]["subject"])

    print("Topic:",
          results["metadatas"][0][i]["topic"])

    print("Content:")
    print(results["documents"][0][i])

    print("-"*60)

Subject: Data Engineering
Topic: APIs and Data Collection
Content:
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
------------------------------------------------------------
Subject: Data Engineering
Topic: SQL Databases
Content:
A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
------------------------------------------------------------
Subject: Python Programming
Topic: Pandas Library
Content:
Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.
--------

In [10]:
query = "How can machine learning understand language?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for doc in results["documents"][0]:
    print(doc)
    print()

A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generate human-like text answer questions summarize documents and perform many language tasks. Examples include GPT Claude and LLaMA.

Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.

Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R-squared for regression. A good model generalizes well to new data it has not seen before.



In [11]:
#filter by subject
query = "How do chatbots generate responses?"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"subject":"Artificial Intelligence"}
)

for doc in results["documents"][0]:
    print(doc)
    print()

In [12]:
#keyword search function
def keyword_search(query):

    query = query.lower()

    matches = []

    for _, row in df.iterrows():

        if query in row["content"].lower():

            matches.append({
                "topic": row["topic"],
                "content": row["content"]
            })

    return matches

In [13]:
#Compare Keyword vs Semantic Search
query = "systems that learn patterns from data"

In [14]:
print("KEYWORD SEARCH")
print("="*50)

keyword_results = keyword_search(query)

print(keyword_results)

KEYWORD SEARCH
[]


In [15]:
print("SEMANTIC SEARCH")
print("="*50)

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i in range(len(results["documents"][0])):

    print("Topic:",
          results["metadatas"][0][i]["topic"])

    print()

SEMANTIC SEARCH
Topic: Supervised Learning

Topic: Decision Trees

Topic: Data Visualization



In [16]:
#create results dataframe
query = "database management"

results = collection.query(
    query_texts=[query],
    n_results=5
)

output = pd.DataFrame({
    "Topic":[m["topic"] for m in results["metadatas"][0]],
    "Subject":[m["subject"] for m in results["metadatas"][0]],
    "Content":results["documents"][0]
})

output

,Topic,Subject,Content
0,SQL Databases,Data Engineering,A database is an organized collection of data ...
1,Data Cleaning,Data Engineering,Data cleaning involves fixing or removing inco...
2,Supervised Learning,Machine Learning,Supervised learning is a type of machine learn...
3,Decision Trees,Machine Learning,A decision tree is a machine learning model th...
4,Large Language Models,Generative AI,A Large Language Model or LLM is an AI model t...
